# পাঠ ১৮ (পরবর্তী): রসিদ যা প্রমাণ করে একটি *মানব* কর্তৃপক্ষ অনুমোদিত কাজটি

এই পাঠটি প্রমাণ করে **এজেন্ট** কী করেছিল এবং **গেট** কী সিদ্ধান্ত নিয়েছিল। এই নোটবুকটি হারানো অর্ধেক যোগ করে: প্রমাণ যে একটি **নির্দিষ্ট মানুষ** **নির্দিষ্ট** কাজটি অনুমোদন করেছে — একটি পৃথক, মানুষের-ধারিত স্বাক্ষর সম্পূর্ণ মানক কাজের উপরে, যা অফলাইনে যাচাই করা হয়।

এখানে উভয় কৃত্রিম বস্তু ব্যবহার করে **পাঠের রসিদের মতোই এনভেলপ আকার**: একটি ফ্ল্যাট পেলোড যার একটি `type` ক্ষেত্র রয়েছে, Ed25519 দ্বারা স্বাক্ষরিত যা ক্যনোনিকাল পেলোড বাইটের SHA-256 এর উপর ভিত্তি করে, যার সাথে একটি গঠনমূলক `signature` অবজেক্ট সংযুক্ত (এবং স্বাক্ষরিত বাইট থেকে বাদ দেওয়া)। অনুমোদনের রসিদ একটি নতুন `type` (`human.approval.v1`) কাজের টাইপের পাশাপাশি, তাই একটি `verify_chain` উভয় ধরণের কৃত্রিম বস্তু একই কোড পাথ দ্বারা পরীক্ষা করে যা আপনি মূল নোটবুকে তৈরি করেছিলে। এটি ইন্টারনেট-ড্রাফ্টের কো-স্বাক্ষর পাথের আকারও (draft-farley-acta-signed-receipts) যা পাঠ অনুসরণ করে।

মূল নোটবুকের ডেমো ভেরিফায়ারের থেকে একটি জানানোর আপগ্রেড: এখানে ভেরিফায়ার `signature.key_id` একটি **পিন করা কী রেজিস্ট্রি** এর বিরুদ্ধে সমাধান করে, রসিদের ভিতরে থাকা পাবলিক কি-তে বিশ্বাস না করে। এটাই উৎপাদন পদ্ধতি যা পাঠের নিজস্ব চেকলিস্ট সুপারিশ করে ("যাচাই পাবলিক কী প্রকাশ করুন"), এবং এটি ভুয়া স্বাক্ষরকে প্রত্যাখ্যান বানায়, নিজের কি নিয়ে বাইপাস হওয়ার বদলে।

এই নোটবুক শেখায় নিয়ম: **একটি স্বাক্ষরিত অনুমোদন নিজেই কর্তৃত্ব নয়।** কর্তৃত্ব তখনই বিদ্যমান যখন অনুমোদন রসিদ এবং কাজের রসিদ একই মানক কাজের সাথে কার্যনির্বাহী সময়ে এখনও যুক্ত থাকে, এমন একটি নীতি সংস্করণ, কী, এবং মেয়াদ যা এখনও বৈধ, এবং এমন একটি অনুমোদন যা ইতিমধ্যে ব্যবহৃত হয়নি। প্রত্যেক ব্যর্থতা একটি **স্বতন্ত্র কারণ** ব্যাখ্যা করে, তাই আপনি বলতে পারবেন *কর্তৃত্ব অবসান হয়েছে* না *কার্যকর কাজ পরিবর্তিত হয়েছে*।


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## সঠিক ক্রিয়া

অনুমোদনের একক হল **ক্যানোনিক্যাল অ্যাকশন অবজেক্ট** — "রিফান্ড অনুমোদন করুন" এর মতো অস্পষ্ট লেবেল নয়, বরং সুনির্দিষ্ট, সম্পূর্ণ নির্দিষ্ট ক্রিয়া। পুরো অবজেক্টটির উপর সাইন করা (এবং এটি থেকে একটি ডাইজেস্ট তৈরি করা) আমাদের পরে প্রমাণ করতে দেয় যে মানুষ এটি অনুমোদন করেছেন *এটা* এবং অন্য কিছু নয়।


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## এক খাম, দুই কর্তৃপক্ষ

প্রতিটি রসিদ হলো পাঠের খাম: একটি ফ্ল্যাট পে'লড যার একটি `type` ফিল্ড থাকে, সঙ্গে একটি `signature` অবজেক্ট (`alg`, `sig`, `key_id`) যা সই করা বাইটগুলোর অংশ **নয়**। `verify_envelope` হলো উভয় রসিদের জন্য ভাগ করা গঠনগত + স্বাক্ষর যাচাই; এটি কোন **পিন্ড কী রেজিস্ট্রি**-র বিরুদ্ধে `signature.key_id` রেজ়ল্ভ করে সেটাই কর্তৃপক্ষগুলোকে পৃথক রাখে:

- **অনুমোদন রসিদ** (`human.approval.v1`) — নির্দিষ্ট approver, সম্পূর্ণ ক্যানোনিকাল অ্যাকশন **এবং তার ডাইজেস্ট**, `policy_version`, ইস্যু + মেয়াদ শেষের টাইমস্ট্যাম্প। একবারের জন্য ব্যবহারের হিসাব চেইন স্তরে ট্র্যাক করা হয়।
- **অ্যাকশন রসিদ** (`agent.action.v1`) — এজেন্ট পরিচয়, `run_id`, একই ক্যানোনিকাল অ্যাকশন **ডাইজেস্ট**, এক্সিকিউশনের ফলাফল + টাইমস্ট্যাম্প, এবং `parent_approval_ref`: অনুমোদনের `receipt_hash`, যা পাঠের চেইনের `previous_receipt_hash` এর মতই নিয়ম।

ভাগ করা `action_digest` ফিল্ড হলো সেই যোগফল যার ওপর বাইন্ডিং নির্ভর করে। `key_id` শুধুমাত্র লুকআপ হিন্ট হিসেবে সিগনেচার অবজেক্টে থাকে: এটাকে ভিন্ন পিন্ড কী-তে রি-পয়েন্ট করলে সিগনেচার যাচাই ব্যর্থ হয়, তাই এটি কিছুই প্রদান করে না।


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: যেখানে আসলে binding সিদ্ধান্ত নেওয়া হয়

`verify_chain` হল দুইটি সিগনেচার চেকের উপর একটি সুবিধাজনক র‍্যাপার **নয়**। এটি একমাত্র স্থান যেখানে শেয়ার্ড canonical `action_digest`, অনুমোদনের পলিসি/কী/মেয়াদি **তারতাজা অবস্থা**, এবং অনুমোদনের **এককালীন ব্যবহার** একসাথে পরীক্ষা করা হয়, কার্যটি *এখনই* কার্যকর করা হচ্ছে কিনা সেটির বিরুদ্ধে।

প্রতিটি ব্যর্থতা একটি **স্বতন্ত্র কারণ** সহ প্রত্যাখ্যান করে, তাই প্রত্যাখ্যানের পাঠক জানতে পারে কর্তৃত্বটি স্থির হয়েছে কিনা (পলিসি পরিবর্তিত হয়েছে, কী রোটেট হয়েছে, অনুমোদনের মেয়াদ শেষ হয়েছে, অনুমোদন ব্যবহার হয়েছে) অথবা কার্যকর করা কার্যটি এখনো বৈধ অনুমোদনের অধীনে পরিবর্তিত হয়েছে কিনা (ডাইজেস্ট বিনিময়)।


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## বাইনডিং কি ধরে

নিচের প্রতিটি ক্ষেত্রে একটি **স্পষ্ট কারণ**সহ **বন্ধ** হয়। প্রথম ব্লকটি ক্লাসিক সেট (ট্যাম্পার, বিভ্রান্ত ডেপুটি, রিপ্লে, যেকোন ক্ষমতার উপর জালিয়াতি, বিকৃত ইনপুট)। দ্বিতীয় ব্লকটি সেই জুটি যা বৈশিষ্ট্যটিকে দাবিকৃত নয় বরং বাস্তব করে তোলে:

- **জমে থাকা ক্ষমতা** — সু_signatureঅবৈধ থাকে, কিন্তু নীতি সংস্করণ পরিবর্তিত হয়, অনুমোদক কী পিন্ড রেজিস্ট্রির বাইরে সরানো হয়, অথবা অনুমতি কার্যকর হবার আগে মেয়াদোত্তীর্ণ হয়ে যায়;
- **ডাইজেস্ট প্রতিস্থাপন** — একটি বৈধভাবে স্বাক্ষরিত ক্রিয়াকলাপ রসিদ যার `parent_approval_ref` একটি *বাস্তব* অনুমোদনের দিকে ইঙ্গিত করে, কিন্তু সেই অনুমোদনের প্রচলিত ক্রিয়াকলাপ ডাইজেস্ট বাস্তবে কার্যকর হওয়া ক্রিয়াকলাপের সাথে মিলে না।


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## এটি কী প্রমাণ করে — এবং কী প্রমাণ করে না

**প্রমাণ করে:** একজন নামকরা মানব এটি অনুমোদন করেছে *এই সুনির্দিষ্ট ক্যানোনিক্যাল ক্রিয়াটি* (পূর্ণ ক্রিয়া + ডাইজেস্ট, একটি পিন করা রেজিস্ট্রি থেকে সমাধান করা কী দিয়ে স্বাক্ষরিত), এবং এজেন্ট *একেবারে সেই অনুমোদিত ক্রিয়াটি* কার্যকর করেছে (একই ডাইজেস্ট, রসিদ `receipt_hash` দ্বারা অনুমোদনের সাথে বেঁধে দেওয়া, পাঠের নিজস্ব চেইন রীতি) — যখন অনুমোদনের নীতি সংস্করণ, কী, এবং মেয়াদ এখনও সক্রিয় ছিল, ঠিক একবার। যদি কোনও পক্ষ পরিবর্তন করে, চেইন বন্ধ হয়ে যায়, এবং প্রত্যাখ্যানের কারণ বলে দেয় **কোন** বৈশিষ্ট্যটি ভঙ্গ হয়েছে: পুরানো কর্তৃপক্ষ বনাম পরিবর্তিত ক্রিয়া।

**প্রমাণ করে না:** যে অনুমোদন UI মানবকে দেখিয়েছে তারা যা স্বাক্ষর করছেন তা তারা কীভাবে ভাবছিল (WYSIWYS নিজস্ব একটি সমস্যা), যে কীটি রোটেশন এর আগে জোরপূর্বক নেওয়া হয়নি বা চুরি হয়নি, অথবা যে পরবর্তী কার্যকর প্রভাবগুলি ক্রিয়ার সাথে মেলে। স্বাক্ষরিত হওয়া ≠ অনুমোদিত হওয়া: একটি অবৈধ নীতি, একটি রোটেট করা কী, একটি মেয়াদোত্তীর্ণ উইন্ডো, অথবা একটি ভিন্ন ডাইজেস্টের উপর একটি বৈধ স্বাক্ষর এখানেএকথাও দেয় না।

দুটি রসিদ প্রকার পাঠের এনভেলপ এবং একটি `verify_chain` কোড পাথ ভাগ করে উদ্দেশ্যমূলকভাবে: আপনি যা তৈরি করেছেন মূল নোটবুকে ক্রিয়া রসিদের জন্য সেটি একই কোড যা মানবের অনুমোদন পরীক্ষা করে। একটি যাচাইকারী চুক্তি, পৃথক পিন করা কর্তৃপক্ষ, ক্যানোনিক্যাল ক্রিয়ার ডাইজেস্ট দ্বারা যুক্ত এবং আরও কিছু নয়।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
